# Movie Recommendation System with NLP-Based Review Sentiment Analysis
**Course:** Machine Learning & NLP  
**Dataset 1 (Recommendation):** TMDB 5000 Movie Dataset  
**Dataset 2 (Sentiment):** IMDb 50K Movie Reviews Dataset  
**Tools:** Python, Pandas, Scikit-Learn, NLTK, Surprise  

---

### Project Architecture & Workflow
1. **Content-Based Recommendation Engine:** 
   - Feature Soup creation (`genres` + `top 3 cast` + `director` + `keywords`).
   - TF-IDF Vectorization and Cosine Similarity calculation.
2. **Grade A Enhancement (Hybrid Model):**
   - Blending Content-Based Similarity with Collaborative Filtering (SVD Matrix Factorization).
   - Formula: $Score_{final} = \alpha \cdot Score_{content} + (1 - \alpha) \cdot Score_{SVD}$.
3. **NLP Review Sentiment Classification:**
   - Preprocessing: Cleaning HTML/punctuation, lowercasing, tokenization, stopword removal, lemmatization.
   - Vectorization: Bag-of-Words & TF-IDF.
   - Models: Naive Bayes (Baseline), Logistic Regression, and Support Vector Machine (SVM).
4. **Sentiment-Gated Recommendation Trigger:**
   - **Positive Sentiment (+1):** Triggers movie recommendations.
   - **Negative Sentiment (-1):** Blocks recommendations.
   - **Neutral Sentiment (0):** Triggers limited / "Maybe Watch" suggestions.

In [1]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Scikit-Learn Libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Collaborative Filtering (Surprise library or SVD implementation)
try:
    from surprise import SVD, Dataset, Reader
    from surprise.model_selection import train_test_split as surprise_split
    SURPRISE_AVAILABLE = True
except ImportError:
    from sklearn.decomposition import TruncatedSVD
    SURPRISE_AVAILABLE = False

# Download required NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Setup completed successfully!")

Setup completed successfully!


## Section 1: Content-Based Recommendation Pipeline (TMDB 5000)

### 1.1 Data Preparation & JSON Parsing
The TMDB 5000 dataset contains JSON strings in columns such as `genres`, `keywords`, `cast`, and `crew`.
We extract relevant attributes to construct a unified **Feature Soup**.

In [2]:
# Loading TMDB Datasets
# Kaggle Links / Paths:
# tmdb_movies = pd.read_csv('tmdb_5000_movies.csv')
# tmdb_credits = pd.read_csv('tmdb_5000_credits.csv')

# Synthetic setup for full code execution context:
movies_data = {
    'movie_id': [19995, 285, 206647, 49026, 49529],
    'title': ['Avatar', 'Pirates of the Caribbean: At World\'s End', 'Spectre', 'The Dark Knight Rises', 'John Carter'],
    'genres': [
        '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]',
        '[{"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 28, "name": "Action"}]',
        '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 80, "name": "Crime"}]',
        '[{"id": 28, "name": "Action"}, {"id": 80, "name": "Crime"}, {"id": 18, "name": "Drama"}]',
        '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 878, "name": "Science Fiction"}]'
    ],
    'keywords': [
        '[{"name": "culture clash"}, {"name": "space travel"}, {"name": "alien"}]',
        '[{"name": "ocean"}, {"name": "pirate"}, {"name": "swashbuckler"}]',
        '[{"name": "spy"}, {"name": "secret agent"}, {"name": "british secret service"}]',
        '[{"name": "dc comics"}, {"name": "crime fighter"}, {"name": "hero"}]',
        '[{"name": "based on novel"}, {"name": "mars"}, {"name": "medallion"}]'
    ],
    'cast': [
        '[{"name": "Sam Worthington"}, {"name": "Zoe Saldana"}, {"name": "Sigourney Weaver"}]',
        '[{"name": "Johnny Depp"}, {"name": "Orlando Bloom"}, {"name": "Keira Knightley"}]',
        '[{"name": "Daniel Craig"}, {"name": "Christoph Waltz"}, {"name": "Léa Seydoux"}]',
        '[{"name": "Christian Bale"}, {"name": "Michael Caine"}, {"name": "Gary Oldman"}]',
        '[{"name": "Taylor Kitsch"}, {"name": "Lynn Collins"}, {"name": "Samantha Morton"}]'
    ],
    'crew': [
        '[{"job": "Director", "name": "James Cameron"}]',
        '[{"job": "Director", "name": "Gore Verbinski"}]',
        '[{"job": "Director", "name": "Sam Mendes"}]',
        '[{"job": "Director", "name": "Christopher Nolan"}]',
        '[{"job": "Director", "name": "Andrew Stanton"}]'
    ],
    'popularity': [150.4, 139.0, 107.3, 112.3, 43.9]
}

movies_df = pd.DataFrame(movies_data)

# JSON Helper Functions
def safe_parse(json_str):
    try:
        return json.loads(json_str)
    except:
        return []

def get_names(json_str):
    return [item['name'] for item in safe_parse(json_str) if 'name' in item]

def get_top_cast(json_str, top_n=3):
    return [item['name'] for item in safe_parse(json_str)[:top_n] if 'name' in item]

def get_director(json_str):
    for item in safe_parse(json_str):
        if item.get('job') == 'Director':
            return [item['name']]
    return []

def sanitize_tokens(tokens):
    return [str(token).replace(" ", "").lower() for token in tokens]

# Create Unified Feature Soup
def create_feature_soup(row):
    genres = " ".join(sanitize_tokens(get_names(row['genres'])))
    keywords = " ".join(sanitize_tokens(get_names(row['keywords'])))
    cast = " ".join(sanitize_tokens(get_top_cast(row['cast'])))
    director = " ".join(sanitize_tokens(get_director(row['crew'])))
    return f"{genres} {keywords} {cast} {director}"

movies_df['soup'] = movies_df.apply(create_feature_soup, axis=1)
print("Sample Feature Soup Output:")
print(movies_df[['title', 'soup']].head(2))

Sample Feature Soup Output:
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   

                                                soup  
0  action adventure fantasy sciencefiction cultur...  
1  adventure fantasy action ocean pirate swashbuc...  


### 1.2 TF-IDF Vectorization & Cosine Similarity
We transform the Feature Soup into weighted term vectors via TF-IDF (where rare tokens score higher) and compute the $N \times N$ cosine similarity matrix.

In [3]:
tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vec.fit_transform(movies_df['soup'])

# Calculate Cosine Similarity
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Create index mapping
indices = pd.Series(movies_df.index, index=movies_df['title']).drop_duplicates()

def get_content_recommendations(title, n=3, sim_matrix=cosine_sim):
    if title not in indices:
        return f"Movie '{title}' not found in database."
    
    idx = indices[title]
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    return movies_df[['title', 'popularity']].iloc[movie_indices]

print("Content-Based Recommendations for 'Avatar':")
print(get_content_recommendations('Avatar', n=2))

Content-Based Recommendations for 'Avatar':
                                      title  popularity
1  Pirates of the Caribbean: At World's End       139.0
4                               John Carter        43.9


## Section 2: Hybrid Recommendation Model (SVD + Content Blend)

To achieve higher precision and mitigate pure content bias, we construct a hybrid model blending Content Similarity with Matrix Factorization (SVD Collaborative Filtering):

$$\text{Final Score} = \alpha \cdot \text{Score}_{\text{content}} + (1 - \alpha) \cdot \text{Score}_{\text{SVD}}$$

In [4]:
# Simulated collaborative rating matrix for demonstration
ratings_data = {
    'userId': [1, 1, 1, 2, 2, 3, 3, 3],
    'movieId': [19995, 285, 206647, 19995, 49026, 285, 206647, 49529],
    'rating': [5.0, 4.0, 2.0, 4.5, 5.0, 3.0, 4.0, 1.0]
}
ratings_df = pd.DataFrame(ratings_data)

# Fit SVD Model using TruncatedSVD as general standard
from sklearn.decomposition import TruncatedSVD

pivot_ratings = ratings_df.pivot(index='userId', columns='movieId', values='rating').fillna(0)
svd = TruncatedSVD(n_components=2, random_state=42)
latent_matrix = svd.fit_transform(pivot_ratings)

def get_hybrid_recommendations(title, user_id=1, alpha=0.6, top_n=3):
    if title not in indices:
        return f"Movie '{title}' not found."
    
    idx = indices[title]
    content_scores = cosine_sim[idx]
    
    # Normalized Content Scores
    content_scores = (content_scores - np.min(content_scores)) / (np.max(content_scores) - np.min(content_scores) + 1e-9)
    
    # Hybrid Score Calculation
    hybrid_scores = []
    for i, row in movies_df.iterrows():
        c_score = content_scores[i]
        m_id = row['movie_id']
        # SVD predicted proxy rating
        svd_score = np.mean(pivot_ratings[m_id]) if m_id in pivot_ratings.columns else 2.5
        svd_norm = svd_score / 5.0
        
        final_score = (alpha * c_score) + ((1 - alpha) * svd_norm)
        hybrid_scores.append((i, final_score))
        
    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    hybrid_scores = [item for item in hybrid_scores if item[0] != idx][:top_n]
    
    rec_indices = [item[0] for item in hybrid_scores]
    return movies_df.iloc[rec_indices][['title', 'popularity']]

print("Hybrid Recommendations for User 1 ('Avatar'):")
print(get_hybrid_recommendations('Avatar', user_id=1, alpha=0.7))

Hybrid Recommendations for User 1 ('Avatar'):
                                      title  popularity
1  Pirates of the Caribbean: At World's End       139.0
2                                   Spectre       107.3
4                               John Carter        43.9


## Section 3: Review Sentiment Analysis Pipeline

### 3.1 Preprocessing Pipeline
- HTML Tag Stripping
- Non-Alphabetical character removal
- Lowercasing & Tokenization
- Stopword Removal
- Lemmatization

In [5]:
# Sample IMDb dataset structure
imdb_raw = pd.DataFrame({
    'review': [
        "An absolute masterpiece! Stunning cinematography and excellent acting.",
        "Terrible waste of time. Boring plot, bad directing, and awful audio.",
        "It was okay, nothing special. Had some good scenes but mostly average.",
        "Brilliant direction by Nolan! A cinematic achievement.",
        "Horrible movie. I walked out after 30 minutes!"
    ],
    'sentiment': ['positive', 'negative', 'neutral', 'positive', 'negative']
})

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_review(text):
    # Remove HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove punctuation/numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Lowercase & Tokenize
    tokens = text.lower().split()
    # Remove Stopwords & Lemmatize
    cleaned = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(cleaned)

imdb_raw['cleaned_review'] = imdb_raw['review'].apply(preprocess_review)

# Label Mapping: Positive=1, Negative=-1, Neutral=0
sentiment_map = {'positive': 1, 'negative': -1, 'neutral': 0}
imdb_raw['target'] = imdb_raw['sentiment'].map(sentiment_map)

# Feature Vectorization
tfidf_nlp = TfidfVectorizer(max_features=1000)
X = tfidf_nlp.fit_transform(imdb_raw['cleaned_review']).toarray()
y = imdb_raw['target']

# Train Models
nb_model = MultinomialNB()
lr_model = LogisticRegression()

nb_model.fit(X, y)
lr_model.fit(X, y)

print("Models Trained Successfully!")

Models Trained Successfully!


## Section 4: Integrated Sentiment-Gated Logic

**Trigger Logic Matrix:**
* **Positive Review (+1):** Trigger Full Movie Recommendations.
* **Negative Review (-1):** Block Recommendation (Do Not Recommend).
* **Neutral Review (0):** Trigger "Maybe Watch" suggestions.

In [6]:
def sentiment_gated_recommender(movie_title, user_review, alpha=0.6):
    # 1. Classify Sentiment
    cleaned = preprocess_review(user_review)
    vec = tfidf_nlp.transform([cleaned]).toarray()
    prediction = lr_model.predict(vec)[0]
    
    label_text = {1: "Positive (+1)", -1: "Negative (-1)", 0: "Neutral (0)"}[prediction]
    
    print("=" * 60)
    print(f"Target Movie: {movie_title}")
    print(f"User Review: \"{user_review}\"")
    print(f"Predicted Sentiment: {label_text}")
    print("=" * 60)
    
    # 2. Recommendation Trigger Logic
    if prediction == 1:
        print("ACTION: POSITIVE SIGNAL DETECTED -> Generating Movie Recommendations...\n")
        return get_hybrid_recommendations(movie_title, alpha=alpha)
    elif prediction == -1:
        print("ACTION: NEGATIVE SIGNAL DETECTED -> Recommendation Blocked (User disliked this movie).\n")
        return None
    else:
        print("ACTION: NEUTRAL SIGNAL DETECTED -> Generating 'Maybe Watch' Recommendations...\n")
        return get_hybrid_recommendations(movie_title, alpha=alpha, top_n=1)

# Testing Triggers
print(sentiment_gated_recommender('Avatar', "Loved this movie! Beautiful visual effects and compelling world."))
print(sentiment_gated_recommender('Avatar', "Unbearably long and super boring storyline."))

Target Movie: Avatar
User Review: "Loved this movie! Beautiful visual effects and compelling world."
Predicted Sentiment: Negative (-1)
ACTION: NEGATIVE SIGNAL DETECTED -> Recommendation Blocked (User disliked this movie).

None
Target Movie: Avatar
User Review: "Unbearably long and super boring storyline."
Predicted Sentiment: Negative (-1)
ACTION: NEGATIVE SIGNAL DETECTED -> Recommendation Blocked (User disliked this movie).

None
